In [1]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.io.fits as fits

- Example relevant readings: 
    - https://iopscience.iop.org/article/10.1086/133722 
    - https://iopscience.iop.org/article/10.1086/132961 
    - https://arxiv.org/html/2508.05862v1
    

### Wavelength calibration, a quick overview:

- spectroscopy data have wavelength info 

<img src="hst_hnpegb.png" width=450> (Cloud Atlas observation of HNPegb)

- part of the analysis process is to convert a pixel number to an actual wavelength






In [ ]:
hnpeg = np.loadtxt('HNPEGB.txt', skiprows =16 )

plt.plot( hnpeg[:,0], hnpeg[ :,1] )
dm = plt.title('HST spectrum of HN Peg b')

- important step; we need to know that pixel column of so-and-so detector corresponds to a real wavelength of such-and-such nm/$\mu$m/....
- typically done by looking at a reference spectrum with known lines at known wavelengths


## Wavelength Calibration, the basics: 
- Our goal: know the wavelength, $\lambda$, falling on each pixel $(x,y)$ $\rightarrow$ find $f(x,y)=\lambda$
    - For unresolved sources, want $f(y)=\lambda$ *for that observation*


- Dispersion on array usually close to linear, but not perfect
- might have offsets in starting $\lambda$ per frame and along slit 

</br>

- Take spectra of a source with known spectral lines
  - Could be lamp, sky, or lines in object spectrum
  - capability that's internal to the instrument usually; the light source is an excited atomic gas, so the spectrum is a bunch of emission lines 
    - electric current excites the gas
    - emit photons of particular wavelengths

  - Lamp usually He, Ne, Ar, Xe (lines very precisely and accurately known)
  
  - Ar is a good option to use since it has lots of electrons and so lots of electronic transitions 
    - The more emission lines, the better you can do at calibrating the wavelengths of all the pixels
    - also a noble gas so it won't react to contaminants 
    - there's no shortage of argon in the world (it's 1% of Earth's atmosphere)
    - one of the most heavily used spectrographs in the world at a professional facility 
    
    
### Common calibration sources:

|Source                      | Wavelength      | Resolution                  | Doppler | Slit Shifts | Extra Time |
|:--------------------------:|:---------------:|:---------------------------:|:-------:|:-----------:|:----------:|
|Object                      | any             | depends on object           | incl.   | incl.       | no         |
|OH$^-$ sky lines            | near IR         | R$\gt$600 to resolve blends | no      | no          | no         |
|Lamp: He, Ne, Ar, Xe        | UV--near IR     | all, watch blends           | no      | no          | yes        |
|Planetary nebulae (H, He)   | visible--mid IR | low only (few  lines)       | extra   | maybe       | yes        |
|Star lines                  |  UV, near IR    | depends on line             | extra?  | extra       | yes        |
|Telluric lines in starlight |  near IR        | depends on line             | no      | extra       | yes        |

Telluric = from the Earth ('s atmosphere)



- Identify line locations on the frame per row, or in spectrum $\rightarrow$ "draw table" of pix *vs.* $\lambda$
- Fit some function (usually polynomials) to those locations "draw plot" $\rightarrow$ map out the array or spectrum
- For best fit 
    - need many more lines than degree of polynomial
    - want some lines near each end of spectrum to avoid extrapolation
    - lines must be distinct for a good fit
    - lines must be narrow enough for project goals

- Is it always just a simple linear function?
    - No: cross-dispersed spectra are 1 example:
        - Dispersion is usually close to linear per row for gratings
        - But, zero of each row may differ (``smile'')





- Stars shift in slit frame-to-frame!
    - Shift wavelength scale by fractional pixels to correct

## Example Using OH$^-$ Lines: 

In [ ]:
# The steps:

# ** Read data
# ** Extract column, plot
# ** Estimate location of spectral lines
# ** Refine estimates with a centroid fit
# ** Identify lines in a line atlas
# ** Do polynomial fit
# ** Plot and evaluate residuals
# ** Shift line list by 1 to show a bad fit
# ** Compare significance of coefficients

# ** Read data
# It's the same data as previous lecture's
lsdat1 = fits.getdata('w00846.fit')
lsdat2 = fits.getdata('w00848.fit')
lsdat1.shape = lsdat1.shape[1:]
lsdat2.shape = lsdat2.shape[1:]
reddat = lsdat1 - lsdat2

In [ ]:
dm = plt.imshow( reddat , origin = 'lower')

In [ ]:
# Say we want to calibrate the left spectrum.  Sky lines for that are in lsdat2
dm = plt.imshow( np.clip(lsdat2, 1e-38, 1000), cmap ='RdYlBu' ,origin='lower')

In [ ]:
#start by getting the sky
ny, nx = lsdat2.shape
print(ny,nx)

# ** Extract column, plot
cx = 92


plt.title('Sky Column at Spectrum Position')
plt.xlabel('Pixel')
plt.ylabel('Sky Intensity')
dm = plt.plot(lsdat2[:,cx])


In [ ]:
#%matplotlib inline

In [ ]:
# Bad pixels dominate.  Best to do a median filter.

dx   = 3
spec = lsdat2[:, cx-dx:cx+dx+1] #why +1?
sky  = np.median(spec.T, axis=0)
xsky = np.arange(sky.size, dtype=float)

#plt.clf()
plt.title('Sky Median at Spectrum Position')
plt.xlabel('Pixel')
plt.ylabel('Sky Intensity')
dm = plt.plot(xsky, sky)


In [ ]:
# ** Estimate location of spectral lines

# read off image with plot for help
linest = np.array([43, 55, 78, 89, 126, 144, 160, 201]) # integer pixel indices
linecent = np.array(linest, dtype=float)

# ** Refine estimates with a line-center fit
dy = 7

i = 0
# pull up gaussfit help page
import gaussian as G
# help(G)
# build up a routine core:
# chunk out a piece of a sky line in y and x
ypiece =  sky[linest[i] - dy : linest[i] + dy + 1]
xpiece = xsky[linest[i] - dy : linest[i] + dy + 1]
# look at the numbers
#plt.clf()
plt.title('Sky Emission Line')
plt.xlabel('Pixel')
plt.ylabel('Sky Intensity')
dm = plt.plot(xpiece, ypiece)


In [ ]:
# Refine estimates with a centroid fit
# Gaussian fit to line

plt.title('Sky Emission Line')
plt.xlabel('Pixel')
plt.ylabel('Sky Intensity')
plt.plot(xpiece, ypiece)

gausscoefs = G.fitgaussian(ypiece, xpiece)
dm = plt.plot(xpiece, G.gaussian(xpiece, *gausscoefs[0:3]))
print(gausscoefs)

In [ ]:
# hmm, that width looks very wide...but center is good
# Ah!  The background is not zero.  Try subtracting it off (could also fit it).
gausscoefs = G.fitgaussian(ypiece - ypiece.min(), xpiece)
# plt.clf()
plt.title('Sky Emission Line')
plt.xlabel('Pixel')
plt.ylabel('Sky Intensity')
plt.plot(xpiece, G.gaussian(xpiece, *gausscoefs[0:3]) + ypiece.min())
dm = plt.plot(xpiece, ypiece)

# note that fitted centers are pretty close to each other
print(gausscoefs)

In [ ]:
#repeat for all lines
for i in np.arange(linest.size):
    ypiece =  sky[linest[i] - dy : linest[i] + dy + 1]
    xpiece = xsky[linest[i] - dy : linest[i] + dy + 1]
    gausscoefs = G.fitgaussian(ypiece - ypiece.min(), xpiece)
    linecent[i] = gausscoefs[1]

# check those:
print(f"Line centers: {linecent}")
print(f"Line estimates: {linest}")
print(f"Differences: {linecent-linest}")

In [ ]:
# whoa, line 2 is out of whack, what happened?
i=2
ypiece =  sky[linest[i] - dy : linest[i] + dy + 1]
xpiece = xsky[linest[i] - dy : linest[i] + dy + 1]

plt.title('Sky Emission Line')
plt.xlabel('Pixel')
plt.ylabel('Sky Intensity')
dm = plt.plot(xpiece, ypiece)


In [ ]:
gausscoefs = G.fitgaussian(ypiece - ypiece.min(), xpiece)
print(gausscoefs)

plt.plot(xpiece, G.gaussian(xpiece, *gausscoefs[0:3]) + ypiece.min())
dm = plt.plot(xpiece, ypiece)


In [ ]:
#  no line.  Always check your work.  Looking at the image, 
# the line is pretty faint.  Get rid of it and redo:

linest = np.array([43, 55, 89, 126, 144, 160, 201])
linecent = np.array(linest, dtype=float)

dy = 7

for i in np.arange(linest.size):
    ypiece =  sky[linest[i] - dy : linest[i] + dy + 1]
    xpiece = xsky[linest[i] - dy : linest[i] + dy + 1]
    gausscoefs = G.fitgaussian(ypiece - ypiece.min(), xpiece)
    linecent[i] = gausscoefs[1]

print(f"Line centers: {linecent}")
print(f"Line estimates: {linest}")
print(f"Differences: {linecent-linest}")

In [ ]:
# ** Identify lines in a line atlas

# Unix:
# less ohsky/README
# less ohsky/database.txt
# eog ohsky/*.jpg 
# compare to display and database, starting with line at 1.76484



In [ ]:
# first attempt to identify wavelengths (in microns)
wavl = np.array([1.68359, 1.68991, 1.69504, 1.70041, 1.7119, 1.72056, 1.76484])
#                42.909  55.2855  89.7816  127.360 145.168  160.533  201.090

# ** Do polynomial fit
# NOTE: VERY IMPORTANT!  *ALWAYS* do any fitting in double precision.
# Fitting involves taking the difference between a model and
# data.  When the fit is looking good, the differences will be small,
# sometimes near machine precision.

deg = 4
wfit = np.polyfit(linecent, wavl, deg)
print(wfit)

In [ ]:
# occasionally Numpy is irrational...
wfit = wfit[::-1]  # reverse it using a negative stride
print(wfit)
#array([  1.62018401e+00,
#         2.62198273e-03,
#         -3.37330983e-05,
#         1.80490148e-07,
#         -2.97445864e-10])

In [ ]:
# ** Plot and evaluate residuals

# first plot the whole thing
waveln = 0
for i in np.arange(wfit.size):
    waveln += wfit[i] * xsky**i

# plt.clf()
plt.title('Line Center Fit')
plt.xlabel('Pixel')
plt.ylabel('Wavelength (um)')
plt.plot(xsky, waveln)
dum= plt.plot(linecent, wavl, '+')


In [ ]:
# Something is not right.  Not nearly linear.

# now do formal residuals
waveln = 0
for i in np.arange(wfit.size):
    waveln += wfit[i] * linecent**i

# plt.clf()
plt.title('Line Center Fit Residuals')
plt.xlabel('Pixel')
plt.ylabel('Residual (um)')
dm= plt.plot(linecent, wavl-waveln)


In [ ]:
# ** Redo choice of sky lines

wavl = [1.70737,  1.7119, 1.72439, 1.73819, 1.74452, 1.75011, 1.76484]
#        42.909  55.2855  89.7816  127.360  145.168  160.533  201.090
deg = 4
wfit = np.polyfit(linecent, wavl, deg)
wfit = wfit[::-1]  # reverse it using a negative stride
print(wfit)
#array([  1.69525408e+00,
#         2.07145618e-04,
#         2.39304469e-06,
#        -1.47609509e-08,
#         3.12805749e-11])
dummy = 0

In [ ]:
# ** Plot and evaluate residuals

# first plot the whole thing
waveln = 0
for i in np.arange(wfit.size):
    waveln += wfit[i] * xsky**i

# plt.clf()
plt.title('Line Center Fit')
plt.xlabel('Pixel')
plt.ylabel('Wavelength (um)')
plt.plot(xsky, waveln)
dum= plt.plot(linecent, wavl, '+')


In [ ]:
# now do formal residuals
waveln = 0
for i in np.arange(wfit.size):
    waveln += wfit[i] * linecent**i

# plt.clf()
plt.title('Line Center Fit Residuals')
plt.xlabel('Pixel')
plt.ylabel('Residual (um)')
dm= plt.plot(linecent, wavl-waveln)


In [ ]:
# Much better!  Max residual is 0.00008 microns, and they're scattered
# about zero, rather than trending one way or the other.

# Note that we had to do residuals even to see the errors on our scale.

# Can we improve it any?

deg = 3
wfit = np.polyfit(linecent, wavl, deg)
wfit = wfit[::-1]  # reverse it using a negative stride
print(wfit)
#array([  1.69134619e+00,
#         3.80141008e-04,
#         -1.57697312e-07,
#         4.12734205e-10])

In [ ]:
# ** Plot and evaluate residuals

# first plot the whole thing
waveln = 0
for i in np.arange(wfit.size):
    waveln += wfit[i] * xsky**i

# plt.clf()
plt.title('Line Center Fit')
plt.xlabel('Pixel')
plt.ylabel('Wavelength (um)')
plt.plot(xsky, waveln)
dm = plt.plot(linecent, wavl, '+')


In [ ]:
# now do formal residuals
waveln = 0
for i in np.arange(wfit.size):
    waveln += wfit[i] * linecent**i

# plt.clf()
plt.title('Line Center Fit Residuals')
plt.xlabel('Pixel')
plt.ylabel('Residual (um)')
plt.plot(linecent, wavl-waveln)
dummy = 0

In [ ]:
# Even better!  Max residual is 0.00018 microns, but it's *straight*
# beyond the data.  The lower-order fit allows better estimation
# because the difference between the number of points fit (7) and the
# number of parameters (4-5) is larger.  This difference is the number
# of degrees of freedom in the fit (2 vs. 3 in this case).

# ** Compare significance of coefficients

# One way to check what degree we need is just to look at the
# residuals.  Another is to look at the coefficients.  Recall that
# wfit[0] is the wavelength of pixel 0, then there's a linear term,
# and a quadratic term, and so on.  For the 3rd-order case, we have:

#   1.69134619e+00  wavelength of bottom pixel
#   3.80141008e-04  increase in wavelength per pixel
#  -1.57697312e-07  quadratic curvature
#   4.12734205e-10  cubic curvature

# At pixel 0, all but wfit[0] are zeroed by multiplying by xsky[0]=0.
# At the other end of the array, the coefficients are scaled by their
# respective powers of xsky[255]=255.  We can ask, is the effect of
# the terms diminishing with higher order?  We hope so.  Here is the
# high-order fit:

In [ ]:
deg = 4
wfit = np.polyfit(linecent, wavl, deg)
wfit = wfit[::-1]  # reverse it using a negative stride
print(wfit)
for i in np.arange(wfit.size):
    print(i, wfit[i], 255.**i, wfit[i]*255.**i)

In [ ]:
# The tables show the fit term number, its value, how much that value
# gets multiplied on the top of the array, and the product.  

# 0  1.69525407607              1.0  1.69525407607
# 1  0.000207145617873        255.0  0.0528221325576
# 2  2.39304468617e-06      65025.0  0.155607730718
# 3 -1.47609508815e-08   16581375.0 -0.244756861923
# 4  3.12805748598e-11 4228250625.0  0.132262110201

# The terms at order 2 and higher have roughly comparable
# contributions.  At least they're not growing, but it's not great.

In [ ]:
deg = 3
wfit = np.polyfit(linecent, wavl, deg)
wfit = wfit[::-1]  # reverse it using a negative stride
print(wfit)
for i in np.arange(wfit.size):
    print(i, wfit[i], 255.**i, wfit[i]*255.**i)

In [ ]:
# 0  1.69134619               1.0  1.69134619
# 1  0.000380141007761      255.0  0.0969359569789
# 2 -1.57697311526e-07    65025.0 -0.010254267682
# 3  4.12734204998e-10 16581375.0  0.00684370062839

# What we see from these two fits is that in the 3rd-order fit, the
# terms contribute less and less, while in the 4th-order fit, the last
# 3 terms all contribute comparable amounts.  Usually in a good model
# fit, the higher-order terms contribute less (but this depends on the
# model actually being appropriate to the data).



## Things to correct when doing spectroscopy: 
- All previous array detector effects (bias, dark, sky, flat field)
- Sky emission (continuum and lines, especially OH$^{-}$ lines, particularly R/NIR)
- (if applicable) sky absorption (mainly H$_{2}$O, some CH$_{4}$)
    - for photom. these were just a part of the sky background and instrument throughput
    - Now they vary with wavelength *and* we might care for these species

## How to correct them: 
- Bias, dark current: 
    - dark frames
        - No light enters slit
        - Exposures must match target: wavelength-dependent effect!
            - if long exposure, correct for CR
            
        - make master bias 
- Gain variations: dome/screen flats (sky has lines)
    - make sure to 'map' detector correctly 
    
    - don't forget to mask bad pixels 
    
    - make master flat
    
    - twilight flats can be used to remove residual spatial illumination differences between dome flat and sky exposure
    
- Sky emission (background):
  - Subtract off-target rows (or fit low-order polynomial across target)
  - Extended object: subtract off-target exposure
  
- Sky absorption:
  - Divide by "spectral flat field" (hot, featureless star, e.g., B or A star)
  - Try to do low-, medium-, and high-airmass observations of same star, compute extinction coefficient per wavelength



## Frames to Acquire:
- Target
- Dark
- Dome/instrument flat
- Sky (if necessary)
- Spectral flat


#### intermezzo on sky subtraction:
- assume slit of 1 (spectral) pixel wide and N (spatial) pixels tall. How do you get the net counts? 
    - similar to imaging photometry, take gross counts and subtract off the sky counts

- spectroscopy can use a clever trick to make subtraction of sky easier 
    - typical for NIR
    - can be used for VIS, but less so [sky better-behaved, i.e, less short-term variability, and weaker dependence on the wavelength for the transparency of the atmosphere] 

    - observe your source at two different locations within the slit
    - record a spectrum at each location
        - e.g., point the telescope so that the source is located 1/3 of the way down from the top of the slit (loc "A"); get a spectrum
        - move telescope so that source is now located about 1/3 from the bottom of the slit (loc "B"); get a spectrum
    - at end we have two raw 2D data arrays 
    - clean them up (remove bias, flatten the field), and then subtract one from the other, e.g., A - B 
        - sky's contribution to the source spectrum from location A is in location B's data array and vice versa, the sky's contribution to the source spectrum at location B is in location A's data array
        - subtraction of 2 arrays removes the appropriate sky contribution from each spectrum! 
        - wind up with a negative spectrum for location B, since it is effectively sky contribution - gross counts, but then just multiplying that spectrum by -1 makes it a positive spectrum again.
        
        - primary thing to check is if the OH emission lines are canceling out
            - some changes might be due to OH variation in atm, clouds but if you do this often enough there should be canceling; if not something's wrong
           

- What's left once calibration is done? 
    - you need to convert ADU/pixel into flux per pixel (aka a real spectrum)
    
        - need to extract spectrum and collapse it into a 1-D image of pixel number versus ADU/pixel 
            - simplest (unrealistic) case, the spectrum lies exactly along one row (or column) of the CCD $\rightarrow$ can just extract this row from image
            - IRL spectrum covers more rows (or columns) and the extraction process involves some manner of summing a few adjacent rows perpendicular to the dispersion (an “after-the-fact” pixel binning) at each point along the dispersion
            - often spectra might be curved on detector too
    - Wavelength standard and wavelength calibration lamps (comparison arcs and standard stars)
    - typically with help of premade packages offered from telescope (from [IRAF](https://iraf-community.github.io/)/[pyraf](https://iraf-community.github.io/pyraf.html) )


<img src="bevington_ch6_fig7.png" width=450>
<img src="bevington_ch6_fig8.png" width=450>

(Bev. Ch6 figs 6.7 and 6.8)

### Calibration Observations:
- How often?
    - Often enough to make sure the "solution" (fit) has not changed *significantly*
- Depends on:
  - Instrument mechanics (flexure)
  - Instrument temperature changes
  - Resolution
  - How accurate a calibration the program requires (i.e., what's "significant")
  

In [ ]:
#!/usr/bin/env python3
# Showing spectroscopy files, with examples of data from IRTF SpeX.
# November 21, 2022
# by Yan Fernandez for AST 4762/5765.
"""Showing spectroscopy files, with examples of data from IRTF SpeX."""
#----------------------------------------------------------------------------
#This program just reads in a few example files from IRTF's SpeX
#instrument. I show examples of prism-mode data and LXD-mode data.
#----------------------------------------------------------------------------
#import numpy as np
#import math
#import matplotlib.pyplot as plt
#import sys
#----for reading in and writing out FITS files:
#import astropy
#from astropy import io
#from astropy.io import fits
#----for doing some fitting
#from astropy import modeling
#don't need:
#from numpy import polynomial
#import copy
#import photutils
#from photutils.centroids import centroid_com, centroid_2dg
#from scipy import special
#from scipy import stats
#from scipy.stats import chi2
#----------------------------------------------------------------------------
#plt.close('all')
#----------------------------------------------------------------------------
# read in some prism files and some LXD files.
#prism files -- note the A and B beams.
fil_p1 = 'sbd.2022A069.220325.sn263std.00077.a.fits'
fil_p2 = 'sbd.2022A069.220325.sn263std.00078.b.fits'
#LXD files -- note the A and B beams.
fil_l1 = 'sbd.2022A069.220325.sn263std.00042.a.fits'
fil_l2 = 'sbd.2022A069.220325.sn263std.00043.b.fits'
#open the files, save the relevant info. main data part, not the extensions.
#with fits.open(fil_p1) as p1:
#    p1im1 = p1[0].data
#    p1im2 = p1[1].data
#    p1im3 = p1[2].data
#with fits.open(fil_p2) as p2:
#    p2im1 = p2[0].data
#    p2im2 = p2[1].data
#    p2im3 = p2[2].data
#with fits.open(fil_l1) as l1:
#    l1im1 = l1[0].data
#    l1im2 = l1[1].data
#    l1im3 = l1[2].data
#with fits.open(fil_l2) as l2:
#    l2im1 = l2[0].data
#    l2im2 = l2[1].data
#    l2im3 = l2[2].data

p1im1 = fits.getdata(fil_p1)
p2im1 = fits.getdata(fil_p2)
l1im1 = fits.getdata(fil_l1)
l2im1 = fits.getdata(fil_l2)


#subtract them. A minus B.
p1 = p1im1 - p2im1
l1 = l1im1 - l2im1
#----------------------------------------------------------------------------
#display the prism files
qm = np.median(p1im1)
qs = np.std(p1im1)
plt.figure()
plt.imshow(p1im1, interpolation='None', origin='lower', \
vmin=qm-0.1*qs, vmax=1000, cmap='Blues_r')
plt.xlabel("x")
plt.ylabel("y")
plt.title("Fig. 1: example - prism-mode of SpeX - bright star, A beam")
dm = plt.colorbar()

#plt.show()
qm = np.median(p2im1)
qs = np.std(p2im1)
plt.figure()
plt.imshow(p2im1, interpolation='None', origin='lower', \
vmin=qm-0.1*qs, vmax=1000, cmap='Blues_r')
plt.xlabel("x")
plt.ylabel("y")
plt.title("Fig. 2: example - prism-mode of SpeX - bright star, B beam")
dm = plt.colorbar()
#plt.show()



# subtraction of Figure 2 from Figure 1:

qm = np.median(p1)
qs = np.std(p1)
plt.figure()
plt.imshow(p1, interpolation='None', origin='lower', \
vmin=qm-0.5*qs, vmax=qm+2.5*qs, cmap='Blues_r')
plt.xlabel("x")
plt.ylabel("y")
plt.title("Fig. 3: example - prism-mode of SpeX - bright star, A - B beam")
dm = plt.colorbar()
#plt.show()
#input(":")


In [ ]:
# in prism mode the x-axis is the wavelength axis, all these figures show are just single rows of the array:

#----------------------------------------------------------------------------
#pull out traces from the prism spectra --- just individual rows,
#for simplicity.
#A beam:
abeamonsrc = p1im1[887,800:2048]
abeamonsky = p1im1[962,800:2048]
#B beam
bbeamonsrc = p2im1[962,800:2048]
bbeamonsky = p2im1[887,800:2048]
#x-pixel values. particularly easy for prism-mode.
xpxl = np.linspace(800,2047,2047-800+1)
#display the A beam's spectrum and the B beam's sky.
#I.e.,it's showing the same pixel locations from the 2 separate files.
plt.figure()
plt.plot(xpxl, abeamonsrc, label='A beam source+sky')
plt.plot(xpxl, bbeamonsky, label='B beam sky in that location')
plt.xlabel("x pixel location")
plt.ylabel("counts")
plt.title("Fig. 4: A beam trace through source and sky")
dm = plt.legend(loc='upper right')
#plt.show()
#Now vice versa. B beam's spectrum and A beam's sky.
plt.figure()
plt.plot(xpxl, bbeamonsrc, label='B beam source+sky')
plt.plot(xpxl, abeamonsky, label='A beam sky in that location')
plt.xlabel("x pixel location")
plt.ylabel("counts")
plt.title("Fig. 5: B beam trace through source and sky")
dm = plt.legend(loc='upper right')
#plt.show()


In [ ]:
#Now plot the differences -- notice the flux at the high pixels
#is actually sky flux and so it's no subtracted away.
plt.figure()
plt.plot(xpxl, abeamonsrc - bbeamonsky, label='a beam subtraction', \
color='green')
plt.plot(xpxl, bbeamonsrc - abeamonsky, label='b beam subtraction', \
color='magenta')
plt.xlabel("x pixel location")
plt.ylabel("net counts")
plt.title("Fig. 6: net counts, i.e. sky subtracted")
dm = plt.legend(loc='upper right')

#note that at the large pixel values the sky is bright but has been subtracted away



In [ ]:
#----------------------------------------------------------------------------
# now do it for the long-wavelength cross-dispersed mode (LXD) frames.
# note that thermal emission from the telescope and the environment is significant, 
# far brighter than the counts from the star itself. But the sky-subtraction 
# technique still works well


#show the individual files, and then show the subtracted image.
qm = np.median(l1im1)
qs = np.std(l1im1)
plt.figure()
plt.imshow(l1im1, interpolation='None', origin='lower', \
vmin=qm-0.01*qs, vmax=qm+0.3*qs, cmap='Blues_r')
plt.xlabel("x")
plt.ylabel("y")
plt.title("Fig. 7: example - LXD long-mode of SpeX - bright star, A beam")
dm = plt.colorbar()
#plt.show()
qm = np.median(l2im1)
qs = np.std(l2im1)
plt.figure()
plt.imshow(l2im1, interpolation='None', origin='lower', \
vmin=qm-0.01*qs, vmax=qm+0.3*qs, cmap='Blues_r')
plt.xlabel("x")
plt.ylabel("y")
plt.title("Fig. 8: example - LXD long-mode of SpeX - bright star, B beam")
dm = plt.colorbar()
#plt.show()
qm = np.median(l1)
qs = np.std(l1)
plt.figure()
plt.imshow(l1, interpolation='None', origin='lower', \
vmin=qm-0.5*qs, vmax=qm+2.5*qs, cmap='Blues_r')
plt.xlabel("x")
plt.ylabel("y")
plt.title("Fig. 9: example - LXD long-mode of SpeX - bright star, A - B beam")
dm = plt.colorbar()
plt.show()
plt.figure()
plt.imshow(l1, interpolation='None', origin='lower', \
vmin=qm-0.01*qs, vmax=qm+0.01*qs, cmap='Blues_r')
plt.xlabel("x")
plt.ylabel("y")
plt.title("Fig. 10: same as Fig. 9 just harder stretch")
plt.colorbar()
#input(":")
plt.show()

#----------------------------------------------------------------------------
#end of program


## SNR for spectroscopy:

- similar to photometry SNR$\sim \sqrt{N}$ for bright sources
- largest noise sources: background sky contamination and how well the data can be flat fielded
- can get SNR for continuum or given line

    - $n_\mathrm{pix}$, used in the SNR calculation determined by the continuum band-pass range over which the SNR is desired, times the finite width of the spectrum on the CCD
        - e.g., CCD spectrograph with image scale of 0.85 $\overset{\circ}A$/pixel and spectrum with 3 pixels width $\rightarrow$ the SNR continuum over a 100$\overset{\circ}A$ band-pass would use $n_\mathrm{pix} = \frac{100}{0.85} 3 \sim$353
        - narrow line with full width 40$\overset{\circ}A$ would use $n_\mathrm{pix} = \frac{40}{0.85} 3 \sim$141
            - higher SNR than continuum
        


### Telluric corrections

- Earth atmosphere has a lot of $N_2$ (78%), a bit less $O_2$ (20%) and even less $Ar$ (0.93%)...everything else is traces ($CH_4$ 0.00018%, $CO_2$ 0.0407%, $H_2O$ 0.4%....)
- depending on observational wavelength some of these (particularly traces) can affect your spectra
- atmospheric extinction (Rayleigh scat, molecular abs, aerosol scat) changes how much light ends on our detector
- dependence on location on Earth/ altitude for some too (why?)


<img src="tellurics_p1.png" width=650> 
<img src="tellurics_p2.png" width=650> 


(Synthetic abs spec of Earth [ [Smette et al 2014](https://www.aanda.org/articles/aa/full_html/2015/04/aa23932-14/aa23932-14.html)] based on annual mean profile for Cerro Paranal)



<img src="MaunaKea_vs_SOFIA.png" width=650> (Mauna Kea at 4.2km (red) vs SOFIA at 12.5km flight altitude (black) from [here](https://link.springer.com/referenceworkentry/10.1007/978-94-007-5618-2_3) )

- when observing solar system objects, exoplanets, brown dwarfs and other objects some of the 'trouble' molecules are actually of interest 
- how do we isolate the Earth atm influence and just get the astronomical signal?
    - easiest solution: go to space!
        - very high competition, too little hours available 
    - use ground based telescopes and find way to remove the effect of our atmosphere (telluric correction)
    

- Telluric division: 
    - A. take the spectrum of interest 
    - B. take the spectrum of a telluric standard; observe as much as possible through the identical slab of atmosphere (both in spatial and temporal sense)
        - often star without strong features in its spectrum, esp in wav area of interest
            - hot (e.g., B SpT) stars
        - can also use solar-type stars as Sun is well characterized at high R
        - lists of telluric standards exist (e.g., [1](https://articles.adsabs.harvard.edu/pdf/1994PASP..106..508M),[2](https://academic.oup.com/mnras/article/203/3/777/994469) )
    - devide A/B
    
    - typically, iterative procedure with gradual adjustment of the $\lambda$ and intensity to match the two before their division
    - IRAF/pyraf have special routines for this
    
    
- Some alternative approaches using modeling (or modeling+obs): 
    - [Cotton et al 2014](https://academic.oup.com/mnras/article/439/1/387/977221): 
        - make spectrum of the telluric lines based on model of Earth's atm at the time and position of the target observation
        - if target has sufficiently different spec than Earth's: fit model to data directly
        - if comparable: observe telluric standard to determine the state of the atmosphere
            - fit model of the telluric atmospheric transmission to match the observed spectrum, fit that to data
    - [Smette et al 2014](https://www.aanda.org/articles/aa/full_html/2015/04/aa23932-14/aa23932-14.html):
        - use an RT code, atm profiles, instrument resp func 
        - fit obs with model with least-squares MPFIT at particular windows$\rightarrow$ get atm effect
        - remove tellurics

In [ ]:
#final project!